In [1]:
import json
import re
import unicodedata

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
with open("dataset_tf_idf.json", "r", encoding="utf-8") as fichier:
    donnees = json.load(fichier)

In [4]:
documents = donnees["documents"]

In [5]:
requete_principale = donnees["requete_principale"]

In [6]:
autres_requetes = donnees["autres_requetes"]

In [7]:
identifiants = []
textes = []

for document in documents:
    identifiants.append(document["id"])
    textes.append(document["texte"])


In [14]:
# Liste simple de mots peu importants pour la recherche
mots_vides = [
    "la", "le", "les",
    "de", "du", "des",
    "un", "une",
    "dans", "sur",
    "notre", "votre",
    "se", "fait",
    "peut", "etre",
    "depuis", "chaque",
    "jour", "pour", "puis"
]


def nettoyer_texte(texte):
    """
    Cette fonction :
    1. met le texte en minuscules ;
    2. supprime les accents ;
    3. supprime la ponctuation.
    """

    # Passage en minuscules
    texte = texte.lower()

    # Suppression des accents
    texte = unicodedata.normalize("NFD", texte)

    texte = "".join(
        caractere
        for caractere in texte
        if unicodedata.category(caractere) != "Mn"
    )

    # Conservation des lettres, chiffres et espaces
    texte = re.sub(r"[^a-z0-9\s-]", " ", texte)

    return texte

In [16]:
vectoriseur = TfidfVectorizer(
    preprocessor=nettoyer_texte,
    stop_words=mots_vides
)

# Apprentissage du vocabulaire sur les cinq documents
matrice_documents = vectoriseur.fit_transform(textes)

# Affichage du vocabulaire retenu
vocabulaire = vectoriseur.get_feature_names_out()

print("=" * 70)
print("VOCABULAIRE TF-IDF")
print("=" * 70)

print(vocabulaire)

VOCABULAIRE TF-IDF
['abonnement' 'annule' 'application' 'billet' 'bus' 'chambre' 'confirmez'
 'consultez' 'espace' 'gare' 'horaires' 'hotel' 'mobile' 'permet' 'proche'
 'propose' 'reservation' 'reserver' 'service' 'train' 'ville' 'voyager']


In [18]:
def rechercher(requete):
    """
    Transforme une requête en vecteur TF-IDF,
    calcule la similarité cosinus et classe
    les documents du plus proche au moins proche.
    """

    print("\n" + "=" * 70)
    print("REQUÊTE :", requete)
    print("=" * 70)

    # Transformation de la requête en vecteur TF-IDF
    vecteur_requete = vectoriseur.transform([requete])

    # Cas où aucun mot de la requête n'existe dans le vocabulaire
    if vecteur_requete.nnz == 0:
        print("Aucune correspondance trouvée.")
        print("Les mots de la requête sont absents du vocabulaire.")
        return

    # Calcul de la similarité entre la requête et chaque document
    similarites = cosine_similarity(
        vecteur_requete,
        matrice_documents
    )[0]

    # Création de la liste des résultats
    resultats = []

    for index in range(len(documents)):
        resultat = {
            "id": identifiants[index],
            "texte": textes[index],
            "score": similarites[index]
        }

        resultats.append(resultat)

    # Classement par score décroissant
    resultats.sort(
        key=lambda resultat: resultat["score"],
        reverse=True
    )

    # Affichage des résultats
    print("\nCLASSEMENT DES DOCUMENTS\n")

    for position, resultat in enumerate(resultats, start=1):
        print(
            f"{position}. {resultat['id']} "
            f"- score : {resultat['score']:.6f}"
        )
        print(f"   {resultat['texte']}\n")

    # Meilleur document
    meilleur_document = resultats[0]

    if meilleur_document["score"] > 0:
        print("Meilleur document :", meilleur_document["id"])
        print(
            "Meilleur score :",
            f"{meilleur_document['score']:.6f}"
        )
    else:
        print("Aucun document ne correspond à la requête.")


In [20]:
rechercher(requete_principale)



REQUÊTE : train billet réservation

CLASSEMENT DES DOCUMENTS

1. D1 - score : 0.614659
   La réservation du billet de train se fait sur notre application mobile.

2. D2 - score : 0.614659
   Le billet de train peut être annulé depuis votre espace de réservation.

3. D5 - score : 0.482595
   Pour réserver un billet de train consultez les horaires puis confirmez votre réservation.

4. D4 - score : 0.114591
   Le service propose une réservation de chambre dans un hôtel proche de la gare.

5. D3 - score : 0.000000
   Un abonnement de bus permet de voyager chaque jour dans la ville.

Meilleur document : D1
Meilleur score : 0.614659


In [21]:
for requete in autres_requetes:
    rechercher(requete)


REQUÊTE : bus abonnement ville

CLASSEMENT DES DOCUMENTS

1. D3 - score : 0.774597
   Un abonnement de bus permet de voyager chaque jour dans la ville.

2. D1 - score : 0.000000
   La réservation du billet de train se fait sur notre application mobile.

3. D2 - score : 0.000000
   Le billet de train peut être annulé depuis votre espace de réservation.

4. D4 - score : 0.000000
   Le service propose une réservation de chambre dans un hôtel proche de la gare.

5. D5 - score : 0.000000
   Pour réserver un billet de train consultez les horaires puis confirmez votre réservation.

Meilleur document : D3
Meilleur score : 0.774597

REQUÊTE : astronomie galaxie
Aucune correspondance trouvée.
Les mots de la requête sont absents du vocabulaire.
